# 06 Finite-Difference Black-Scholes Solver

Benchmark the finite-difference PDE baseline against analytical Black-Scholes prices.


In [ ]:
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%run 00_project_setup_and_shared_functions.ipynb

config = load_config()
rng = set_global_seed(int(config["random_seed"]))
print("Config loaded and deterministic seed set.")


In [ ]:
K = 24000.0; T = 30 / 365; r = 0.065; sigma = 0.18; option_type = "put"
fd = finite_difference_black_scholes_solver(K=K, T=T, r=r, sigma=sigma, option_type=option_type, S_max=3*K, stock_steps=int(config["finite_difference"]["stock_steps"]), time_steps=int(config["finite_difference"]["time_steps"]), method=config["finite_difference"]["method"])
S_grid = fd["S_grid"]
fd_curve = fd["price_grid"]
analytic = black_scholes_price(S_grid, K, T, r, sigma, option_type)
mask = (S_grid > 0.25*K) & (S_grid < 2.0*K)
metrics = price_error_metrics(analytic[mask], fd_curve[mask], S_grid[mask], K, option_type)
save_output(metrics, "06_finite_difference_error_metrics.json")
assert metrics["RMSE"] < 150
print("VALIDATION PASSED: finite-difference curve is close to analytical Black-Scholes on interior grid")
plt.figure()
plt.plot(S_grid[mask], analytic[mask], label="Analytical BS")
plt.plot(S_grid[mask], fd_curve[mask], "--", label="Finite difference")
plt.title("Finite-difference versus analytical Black-Scholes")
plt.xlabel("Index level")
plt.ylabel("Put value")
plt.legend()
save_current_figure("06_finite_difference_vs_analytical.png")
plt.figure()
plt.plot(S_grid[mask], fd_curve[mask] - analytic[mask])
plt.axhline(0, color="black", linewidth=1)
plt.title("Finite-difference pricing error")
plt.xlabel("Index level")
plt.ylabel("FD - analytical")
save_current_figure("06_finite_difference_error_plot.png")
save_table(pd.DataFrame({"S": S_grid, "finite_difference": fd_curve, "analytical": analytic}), "06_finite_difference_curve.csv")
metrics
